In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [2]:
df = pd.read_csv("sms spam.csv", encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'text']
print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:\n", df['label'].value_counts())

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['label'])

Dataset Shape: (5572, 2)

Class Distribution:
 label
ham     4825
spam     747
Name: count, dtype: int64


In [3]:
x_train, x_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

In [4]:
MAX_WORDS = 5000
MAX_LEN = 50
EMBEDDING_DIM = 64

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train)

x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

x_train_pad = pad_sequences(x_train_seq, maxlen=MAX_LEN)
x_test_pad = pad_sequences(x_test_seq, maxlen=MAX_LEN)

In [5]:
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM),
    GRU(units=64, dropout=0.2, recurrent_dropout=0.2, return_sequences=False),
    Dense(units=32, activation='relu'),
    Dropout(0.3),
    Dense(units=1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 50, 64)              │         320,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru (GRU)                            │ (None, 64)                  │          24,960 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 347,073 (1.32 MB)

 Trainable params: 347,073 (1.32 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
print("\nStarting Model Training...")
history = model.fit(
    x_train_pad, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)


Starting Model Training...
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 90ms/step - accuracy: 0.8866 - loss: 0.3337 - val_accuracy: 0.9574 - val_loss: 0.1458
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - accuracy: 0.9818 - loss: 0.0684 - val_accuracy: 0.9865 - val_loss: 0.0546
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 77ms/step - accuracy: 0.9918 - loss: 0.0290 - val_accuracy: 0.9865 - val_loss: 0.0455
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.9968 - loss: 0.0135 - val_accuracy: 0.9888 - val_loss: 0.0464
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 83ms/step - accuracy: 0.9988 - loss: 0.0067 - val_accuracy: 0.9933 - val_loss: 0.0532
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 115ms/step - accuracy: 0.9990 - loss: 0.0043 - val_accuracy: 0.9888 - val_loss: 0.0445
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.9993 - loss: 0.0022 - val_accuracy: 0.9933 - val_loss: 0.0554
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - accuracy: 1.0000 - loss: 8.

In [ ]:
print("\nEvaluating Model on Test Data...")
loss, accuracy = model.evaluate(x_test_pad, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

y_pred_probs = model.predict(x_test_pad)
y_pred = (y_pred_probs > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))


Evaluating Model on Test Data...
Test Loss: 0.0594
Test Accuracy: 98.74%
 6/35 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [35]:
def predict_multiple_sms(messages_list):
    print("\n--- Batch SMS Prediction ---")
    for msg in messages_list:
        seq = tokenizer.texts_to_sequences([msg])
        padded = pad_sequences(seq, maxlen=MAX_LEN)
        probability = model.predict(padded, verbose=0)[0][0]
        
        if probability > 0.5:
            result = f"SPAM ({probability*100:.1f}%)"
        else:
            result = f"HAM ({probability*100:.1f}%)"
            
        print(f"Message: \"{msg}\" ---> Result: {result}")

In [36]:
test_messages = [
    "CONGRATULATIONS! You won a $1,000 Walmart Gift Card! Click this link to claim it now: http://bit.ly",
    "Hey, are we still meeting for lunch today at 1 PM?",
    "URGENT! Your mobile account has won £2000 cash prize. Call 09066361531 to claim."
]

predict_multiple_sms(test_messages)


--- Batch SMS Prediction ---
Message: "CONGRATULATIONS! You won a $1,000 Walmart Gift Card! Click this link to claim it now: http://bit.ly" ---> Result: SPAM (99.9%)
Message: "Hey, are we still meeting for lunch today at 1 PM?" ---> Result: HAM (0.1%)
Message: "URGENT! Your mobile account has won £2000 cash prize. Call 09066361531 to claim." ---> Result: SPAM (100.0%)


In [37]:
single_test_msg = df['text'].iloc[10]
seq = tokenizer.texts_to_sequences([single_test_msg])
padded = pad_sequences(seq, maxlen=MAX_LEN)
probability = model.predict(padded, verbose=0)[0][0]

if probability > 0.5:
    result = "SPAM"
else:
    result = "HAM"

print(f"Message: \"{single_test_msg}\"")
print(f"Result: {result}")

Message: "I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k? I've cried enough today."
Result: HAM


In [38]:
single_test_msg = df['text'].iloc[5] 

seq = tokenizer.texts_to_sequences([single_test_msg])
padded = pad_sequences(seq, maxlen=MAX_LEN)
probability = model.predict(padded, verbose=0)[0][0]

if probability > 0.5:
    result = "SPAM"
else:
    result = "HAM"

print(f"Message: \"{single_test_msg}\"")
print(f"Result: {result}")

Message: "FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, å£1.50 to rcv"
Result: SPAM
